# 1 NNLS analysis

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.optimize import nnls

In [ ]:
adata_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")
adata_raw

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")
adata

In [ ]:
adata.obs['monocluster'] = adata.obs['10_cluster'].replace({
    '0': 'mono0',
    '1': 'mono1',
    '2': 'mono2',
    '3': 'mono3',
    '4': 'mono4',
    '5': 'mono5',
    '6': 'mono6',
    '7': 'mono7',
    '8': 'mono8',
    '9': 'mono9'
})

sc.pl.umap(adata, color=['monocluster'])

In [ ]:
adata_raw.obs['monocluster'] = adata.obs['monocluster']

In [ ]:
adata_raw.raw = adata_raw
sc.pp.normalize_total(adata_raw)
sc.pp.log1p(adata_raw)

In [ ]:
sc_df = pd.DataFrame(
    adata_raw.raw.X.toarray(),  # sparse → dense
    index=adata_raw.obs_names,
    columns=adata_raw.raw.var_names
)

In [ ]:
ref = sc_df.groupby(adata_raw.obs["monocluster"]).mean()
ref = ref.T

In [ ]:
# bulkdata 불러오기
bulk = pd.read_csv(
    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/04_bulkRNAseq/salmon.merged.gene_tpm.tsv",
    sep="\t",
    index_col=0
)

In [ ]:
bulk

In [ ]:
bulk_mat = bulk.iloc[:, 1:]
bulk_log = np.log2(bulk_mat + 1)

In [ ]:
common_genes = bulk_log.index.intersection(ref.index)

X = ref.loc[common_genes].values        # genes × clusters

In [ ]:
from scipy.optimize import nnls

results = {}

for sample in bulk_log.columns:
    Y = bulk_log.loc[common_genes, sample].values
    
    weights, _ = nnls(X, Y)
    
    proportions = weights / weights.sum()
    
    results[sample] = proportions

# 결과 DataFrame화
prop_df = pd.DataFrame(
    results,
    index=ref.columns
)

prop_df.T.head()

In [ ]:
prop_df.T

# 2. MuSic deconvolution

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from scipy.optimize import nnls

In [ ]:
adata_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")
adata_raw

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")
adata

In [ ]:
adata.obs['monocluster'] = adata.obs['10_cluster'].replace({
    '0': 'mono0',
    '1': 'mono1',
    '2': 'mono2',
    '3': 'mono3',
    '4': 'mono4',
    '5': 'mono5',
    '6': 'mono6',
    '7': 'mono7',
    '8': 'mono8',
    '9': 'mono9'
})

sc.pl.umap(adata, color=['monocluster'])

In [ ]:
adata_raw.obs['monocluster'] = adata.obs['monocluster']

In [ ]:
# counts matrix 추출 (cells × genes)
if "counts" in adata_raw.layers:
    counts = adata_raw.layers["counts"]
else:
    counts = adata_raw.X

# sparse → dense (필요 시)
if not isinstance(counts, np.ndarray):
    counts = counts.toarray()

sc_df = pd.DataFrame(
    counts,
    index=adata_raw.obs_names,
    columns=adata_raw.var_names
)

In [ ]:
adata_raw

In [ ]:
meta = adata_raw.obs[["monocluster", "sample"]].copy()
sc_df["monocluster"] = meta["monocluster"].values
sc_df["sample"] = meta["sample"].values

# donor 단위 평균
cluster_donor_mean = (
    sc_df
    .groupby(["monocluster", "sample"])
    .mean()
)

In [ ]:
cluster_mean = (
    cluster_donor_mean
    .groupby("monocluster")
    .mean()
)

# genes × clusters
X = cluster_mean.T

In [ ]:
# 각 cluster별 gene variance (donor 간 variance)
cluster_var = (
    cluster_donor_mean
    .groupby("monocluster")
    .var()
)

# gene별 평균 variance (cluster 평균)
gene_var = cluster_var.mean(axis=0)

# 0 division 방지
gene_var[gene_var == 0] = 1e-6

weights = 1 / gene_var

In [ ]:
bulk = pd.read_csv(
    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/04_bulkRNAseq/salmon.merged.gene_tpm.tsv",
    sep="\t",
    index_col=0
)
# 첫 annotation column 제거 (필요 시)
bulk_mat = bulk.iloc[:, 1:]

In [ ]:
common_genes = X.index.intersection(bulk_mat.index)

X = X.loc[common_genes]
bulk_mat = bulk_mat.loc[common_genes]
weights = weights.loc[common_genes]

In [ ]:
# \min_{w ≥ 0} ||W^{1/2}(Y - Xw)||^2

results = {}

W_sqrt = np.sqrt(weights.values)

for sample in bulk_mat.columns:
    Y = bulk_mat[sample].values

    # weighted transform
    Yw = Y * W_sqrt
    Xw = X.values * W_sqrt[:, None]

    w, _ = nnls(Xw, Yw)

    # normalize to proportion
    if w.sum() > 0:
        w = w / w.sum()

    results[sample] = w

prop_df = pd.DataFrame(
    results,
    index=X.columns
).T

In [ ]:
prop_df